SIGNAL TEST

The goal of this section is to add a signal component to the model and get an estimate for the posterior distribution p(mu_s|data). After that we shall proceed with model comparison. 

In [1]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.integrate import trapezoid
from scipy.optimize import brentq
from scipy.stats import poisson

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import wimprates as wr

Now, code is imported from the loader

In [2]:
import sys; sys.path.insert(0, '..')
from src.loader import (
    k_obs, b_nominal, s_i,
    roi_mask, ROI,
    reference_cross_section,
    s2_energies, dE, bin_starts, recoil_energy_cutoff_kev, response_matrix_nr,
)

Now we add a signal to the model

We define the joint likelihood function $\mathcal{L}(\text{data} \mid \mu_s, \beta)$ to account for both the dark matter signal strength and the background normalization. However, now we use Bayesian Inference, which treats both the signal strength $\mu_s$ and background normalization $\beta$ as random variables.

We are using the same Region of Interest used for the Frequentist Likelihood (which treated $\mu_s$ as a fixed parameter).

In [3]:
# roi_mask and ROI are imported from loader

def mu_i_combined(mu_s, beta):
    """Expected events per bin within the ROI."""
    return beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]

def log_likelihood_combined(mu_s, beta):
    """Joint log-likelihood for signal strength (μ_s) and background normalization (β)."""
    expected = np.clip(mu_i_combined(mu_s, beta), 1e-12, None)
    return np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

def likelihood_combined(mu_s, beta):
    return np.exp(log_likelihood_combined(mu_s, beta))

BAYES FACTOR

Now that we have a likelihood that is a function of both $\mu_s$ and $\beta$, we can calculate the Bayes factor. For that, we need to calculate the evidence of model 0 (only background) $P(data \mid H_0)$, and of model 1 (background+signal) $P(data \mid H_1)$.

We will calculate the Bayes factor for two different priors: flat (uniform) prior and scale invariant prior (both normalized). In this way, we will study the sensitivity of Bayes factor to prior.

We are starting with a flat prior, and the evidence can be calculated as the integral of the likelihood multiplied by the prior over the parameter space.

In [4]:
# Range definitions
mu_s_min, mu_s_max = 0.001, 100.0  # Start at 0.001 to avoid log(0)
beta_min, beta_max = 0.1, 30.0

mu_s_values = np.linspace(mu_s_min, mu_s_max, 100)
beta_values = np.linspace(beta_min, beta_max, 100)

# ---------------------------------------------------------
# FLAT PRIORS
# ---------------------------------------------------------
# Normalization constants (Area=1)
c_flat_beta = 1.0 / (beta_max - beta_min)
c_flat_mu = 1.0 / (mu_s_max - mu_s_min)

# P(data | H0)_flat (evidence for model 0: mu_s = 0)
likes_H0_flat = np.array([likelihood_combined(0, b) * c_flat_beta for b in beta_values])
P_data_H0_flat = trapezoid(likes_H0_flat, beta_values)

# P(data | H1)_flat (evidence for model 1)
evidence_grid_H1_flat = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_flat[i, j] = likelihood_combined(m, b) * c_flat_beta * c_flat_mu

P_data_H1_flat = trapezoid(trapezoid(evidence_grid_H1_flat, beta_values, axis=1), mu_s_values)

# Bayes factor
BF_01_flat = P_data_H0_flat / P_data_H1_flat
print (f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")

Bayes Factor (Flat Priors): 0.8881


Then, for the scale invariant prior:

In [5]:
# ---------------------------------------------------------
# SCALE INVARIANT PRIORS
# ---------------------------------------------------------
# Normalization constants (Integral of 1/x is ln(x)))
# Integral from a to b of (1/x) dx = ln(b) - ln(a)
c_si_beta = 1.0 / (np.log(beta_max) - np.log(beta_min))
c_si_mu = 1.0 / (np.log(mu_s_max) - np.log(mu_s_min))

def pi_si_beta(b): return c_si_beta / b
def pi_si_mu(m): return c_si_mu / (m)

# P(data | H0)_SI (evidence for model 0: mu_s = 0)
likes_H0_SI = np.array([likelihood_combined(0, b) * pi_si_beta(b) for b in beta_values])
P_data_H0_SI = trapezoid(likes_H0_SI, beta_values)

# P(data | H1)_SI (evidence for model 1)
evidence_grid_H1_SI = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_SI[i, j] = likelihood_combined(m, b) * pi_si_mu(m) * pi_si_beta(b)

P_data_H1_SI = trapezoid(trapezoid(evidence_grid_H1_SI, beta_values, axis=1), mu_s_values)

# Bayes factor
BF_01_SI = P_data_H0_SI / P_data_H1_SI
print(f"Bayes Factor (Scale Invariant): {BF_01_SI:.4f}")

Bayes Factor (Scale Invariant): 0.0225


We can compare both values of the Bayes factor in function of the prior applied:

In [6]:
# Final Comparison
print(f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")
print(f"Bayes Factor (Scale Invariant): {BF_01_SI:.4f}")

Bayes Factor (Flat Priors): 0.8881
Bayes Factor (Scale Invariant): 0.0225


As we can see, we obtained quite different results for each prior, but they both have one thing in common: they are lower than 1. This impplies that the data observed is more likely under the signal hypothesis ($H_1$) than the background-only hypothesis ($H_0$).

Regarding to the prior sensitivity, we can conclude that the evidences of the models (and therefore the Bayes factor) are quite sensitive to the type of prior used. This is because the flat prior penalizes the signal model significantly by spreading probability mass over a large, unsupported range of signal strengths. It states that the signal being 1 is as likely as it being 99; however, the excess of events is actually in smaller values, so the flat prior is not fair with the signal values (gives more importance to higher values than it should). On the other hand, the scale invariant prior is more appropriate when the order of magnitude of the signal is unknown.  It gives equal weight to every decade (e.g., 0.1 to 1 and 1 to 10), making it more "neutral". Consequently, it is more fair with small values of the signal, and it notice that the model $H_1$ fits pretty good the events observed. That is why the scale invariant prior favors much more the model with the dark matter signal over the background-only model.

However, the dramatically low value for $B_{01}$ in the Scale invariant prior case could indicate us that our background model is too rigid. The lack of flexibility could prevent the background from adapting to spectral fluctuations, forcing the Bayesian evidence to favor the model with the dark matter signal just because the background-only model can not properly adapt to the data.

In [7]:
# Systematic uncertainties — defined once, used by both the mass scan and the single-mass limit below
sigma_agg_signal = np.sqrt(
    0.05**2  +   # electron lifetime
    0.025**2 +   # S2 gain g2
    0.12**2  +   # S2 width cut efficiency (dominant for 4 GeV NR)
    0.08**2  +   # radial cut efficiency
    0.07**2  +   # pileup rejection cuts
    0.03**2      # gas event cut
)                # gives sigma_agg_s ~ 16.5%

sigma_agg_background = np.sqrt(
    0.05**2  +   # electron lifetime
    0.025**2     # S2 gain g2
)                # gives sigma_agg_b ~ 5.6%

In [ ]:
# DM masses to scan 
dm_masses = np.array([3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0])  # GeV/c^2

sigma_ref    = 1e-45   # cm^2 — reference cross section
sigma_uppers = []      # will store upper limit at each mass

# Loop over masses and compute upper limit at each
for mw in dm_masses:

    # signal template for this mass
    rate_kev = wr.rate_wimp_std(
        es=s2_energies,
        mw=mw,
        sigma_nucleon=sigma_ref
    )
    rate = rate_kev * dE
    rate = rate * 0.97678  # efficiency factor from paper

    # apply energy cutoff
    rate_cut = rate.copy()
    cutoff_idx = (bin_starts < recoil_energy_cutoff_kev).sum() - 1
    rate_cut[:cutoff_idx] = 0
    suppress = (recoil_energy_cutoff_kev - bin_starts[cutoff_idx]) / bin_starts[cutoff_idx]
    suppress = np.clip(suppress, 0, 1)
    rate_cut[cutoff_idx] *= 1 - suppress

    # project through response matrix
    s_i_mass = rate_cut @ response_matrix_nr

    # roi_mask imported from loader — fixed ROI, ideally optimised per mass
    n_obs_mass    = int(k_obs[roi_mask].sum())
    mu_b_nom_mass = b_nominal.values[roi_mask].sum()
    mu_s_nom_mass = s_i_mass[roi_mask].sum()

    # conservative expectations
    mu_s_cons = max(mu_s_nom_mass * (1 - 1.28 * sigma_agg_signal),     0.0)
    mu_b_cons = max(mu_b_nom_mass * (1 - 1.28 * sigma_agg_background), 0.0)

    # Poisson upper limit 
    def poisson_cdf_minus_010(mu_s_val):
        mu_total = mu_s_val * mu_s_cons + mu_b_cons
        return poisson.cdf(n_obs_mass, mu_total) - 0.10

    try:
        mu_s_up  = brentq(poisson_cdf_minus_010, 0, 1e10)
        # apply floor (limit)
        if mu_s_up * mu_s_cons < 2.3:
            mu_s_up = 2.3 / mu_s_cons if mu_s_cons > 0 else np.nan
        sigma_up = mu_s_up * sigma_ref
    except ValueError:
        sigma_up = np.nan

    sigma_uppers.append(sigma_up)
    print(f"mw = {mw:.1f} GeV/c^2 | mu_s_nom = {mu_s_nom_mass:.5f} | "
          f"n_obs = {n_obs_mass} | sigma_up = {sigma_up:.3e} cm^2")

sigma_uppers = np.array(sigma_uppers)

# Plotting
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(dm_masses, sigma_uppers, 
        color='black', linewidth=2, 
        label='(90% CL)')
ax.fill_between(dm_masses, sigma_uppers, 
                sigma_uppers.max() * 10,
                alpha=0.2, color='gray')

ax.set_yscale('log')
ax.set_xscale('linear')
ax.set_xlabel(r'Dark matter mass [GeV/$c^2$]', fontsize=12)
ax.set_ylabel(r'$\sigma$ [cm$^2$] (SI, spin-independent)', fontsize=12)
ax.set_title('Upper limit on SI DM-nucleon cross section\n'
             '(4 GeV/$c^2$ spin-independent NR, S2-only)', fontsize=11)
ax.set_xlim(2.5, 6.5)
ax.set_ylim(1e-44, 1e-40)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('../plots/upper_limit_cross_section_vs_mass.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
#observed events and background in ROI
#n_obs = the observed count in the ROI
n_obs    = int(k_obs[roi_mask].sum())        
mu_b_nom = b_nominal.values[roi_mask].sum()  
mu_s_nom = s_i[roi_mask].sum()              

# Conservative 10th percentile expectations (sigma_agg_* defined above)
#  s_cons    = s    * (1 - 1.28 * sigma_agg_s)
#  mu_b_cons = mu_b * (1 - 1.28 * sigma_agg_b)
#
# 1.28 is the 90th percentile of the standard normal distribution
mu_s_conservative = mu_s_nom * (1 - 1.28 * sigma_agg_signal)
mu_b_conservative = mu_b_nom * (1 - 1.28 * sigma_agg_background)
mu_s_conservative = max(mu_s_conservative, 0.0)  
mu_b_conservative = max(mu_b_conservative, 0.0) 

# Poisson CDF condition
# This is the left-hand side of Eq.(eq:poisson_ul) minus 0.10:
#
#   f(mu_s) = sum_{k=0}^{n_obs} 
#             e^{-(mu_s*s_cons + mu_b_cons)} * (mu_s*s_cons + mu_b_cons)^k / k!
#             - 0.10
#
# We want to find mu_s_up such that f(mu_s_up) = 0, here the Poisson CDF equals exactly 0.10.
# equivalent to finding where values of mu_s > mu_s_up are excluded at 90% CL.

def poisson_cdf_minus_010(mu_s_val):
    # mu_total is the total expected count at this mu_s value
    mu_total = mu_s_val * mu_s_conservative + mu_b_conservative
    return poisson.cdf(n_obs, mu_total) - 0.10

# Solve numerically using Brent's method
# brentq finds the root of poisson_cdf_minus_010 in [0, 1e9]:
# the value mu_s_up where the CDF = 0.10 exactly — the 90% CL upper limit on mu_s.

try:
    mu_s_upper  = brentq(poisson_cdf_minus_010, 0, 1e9)
    sigma_upper = mu_s_upper * 1e-45  # Eq.(sigma_up = mu_s_up * sigma_ref)

    print(f"\n=== 90% CL Upper Limit ===")
    print(f"mu_s_up                    = {mu_s_upper:.4f}")
    print(f"Expected signal at limit   = {mu_s_upper * mu_s_conservative:.4f} events")
    print(f"sigma_up = mu_s_up * sigma_ref = {sigma_upper:.3e} cm^2")

except ValueError as e:
    print(f"brentq failed: {e}")

# Apply the paper's floor: "never exclude signals with < 2.3 expected events"
# 2.3 is the 90% CL Poisson upper limit for n_obs=0 and mu_b=0. Prevents the limit
# from being artificially strong in bins with very low background.

mu_s_upper_counts = mu_s_upper * mu_s_conservative  # expected signal events at limit

if mu_s_upper_counts < 2.3:
    mu_s_upper  = 2.3 / mu_s_conservative
    sigma_upper = mu_s_upper * 1e-45
    print(f"\nFloor applied: expected signal was {mu_s_upper_counts:.4f} < 2.3 events")

# The 90% CL upper limit on the SI DM-nucleon cross section for a 4 GeV/c^2 WIMP, ROI = 165.3-271.7 PE.
print(f"\n=== Final Result ===")
print(f"mu_s         < {mu_s_upper:.4f}  (dimensionless signal strength)")
print(f"sigma        < {sigma_upper:.3e} cm^2  (90% CL)")
print(f"Signal events at limit: {mu_s_upper * mu_s_conservative:.4f}")


=== 90% CL Upper Limit ===
mu_s_up                    = 372.1458
Expected signal at limit   = 20.4941 events
sigma_up = mu_s_up * sigma_ref = 3.721e-43 cm^2

=== Final Result ===
mu_s         < 372.1458  (dimensionless signal strength)
sigma        < 3.721e-43 cm^2  (90% CL)
Signal events at limit: 20.4941
